# WiDS 2026 | Physics-Gated Multi-Survival Stack

In [1]:
import os
import gc
import math
import json
import time
import random
import warnings
import re
from copy import deepcopy

import numpy as np
import pandas as pd

from scipy.special import expit
from scipy.optimize import minimize
from scipy.stats import rankdata

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAVE_LGB = True
except Exception:
    HAVE_LGB = False

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    HAVE_CAT = True
except Exception:
    HAVE_CAT = False

try:
    import xgboost as xgb
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False


SEEDS = [42, 2026, 3407]
N_SPLITS = 5
HORIZONS = [12, 24, 48, 72]
INTERVALS = [(0, 12), (12, 24), (24, 48), (48, 72)]
EPS = 1e-9


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


def find_data_dir() -> str:
    candidates = [
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/competitions/widsworldwide-globaldathon26",
        "../input/WiDSWorldWide_GlobalDathon26",
        "../input/widsworldwide-globaldathon26",
        ".",
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")) and os.path.exists(os.path.join(c, "test.csv")):
            return c

    roots = ["/kaggle/input", "/kaggle/input/competitions", "../input", "."]
    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            files = set(filenames)
            if {"train.csv", "test.csv"}.issubset(files):
                return dirpath

    raise FileNotFoundError("Could not locate train.csv/test.csv. Please attach the competition dataset to the notebook.")


def safe_div(a, b, default=0.0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full_like(a, default, dtype=float)
    mask = np.abs(b) > EPS
    out[mask] = a[mask] / b[mask]
    return out


def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), 1e-6, 1.0 - 1e-6)


def rank01(x):
    x = np.asarray(x, dtype=float)
    return rankdata(x, method="average") / (len(x) + 1.0)


def discover_legacy_submissions(test_ids):
    roots = ["/kaggle/input", "../input", "."]
    csv_pat = re.compile(r"samplecsv_sub(\d+)\.csv$")
    nb_pat = re.compile(r"final_sub(\d+)_([0-9.]+)\.ipynb$")

    score_by_sub = {}
    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for fn in filenames:
                m = nb_pat.match(fn)
                if m:
                    try:
                        score_by_sub[int(m.group(1))] = float(m.group(2))
                    except Exception:
                        pass

    found = []
    needed = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    test_ids = list(map(str, test_ids))

    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for fn in filenames:
                m = csv_pat.match(fn)
                if not m:
                    continue
                path = os.path.join(dirpath, fn)
                try:
                    df = pd.read_csv(path)
                except Exception:
                    continue
                if not set(needed).issubset(df.columns):
                    continue
                df = df[needed].copy()
                if len(df) != len(test_ids):
                    continue
                if df["event_id"].duplicated().any():
                    continue
                df["event_id"] = df["event_id"].astype(str)
                if set(df["event_id"]) != set(test_ids):
                    continue
                aligned = df.set_index("event_id").loc[test_ids, needed[1:]]
                probs = aligned.values.astype(float)
                if np.any(~np.isfinite(probs)):
                    continue
                probs = np.maximum.accumulate(np.clip(probs, 0.0, 1.0), axis=1)
                sub_num = int(m.group(1))
                found.append({
                    "sub_num": sub_num,
                    "score": score_by_sub.get(sub_num, np.nan),
                    "path": path,
                    "probs": probs,
                })

    if not found:
        return None, {}

    scores = np.array([x["score"] for x in found], dtype=float)
    finite = np.isfinite(scores)
    if finite.any():
        floor = float(np.nanmin(scores[finite]))
        filled = np.where(finite, scores, floor - 0.003)
        weights = np.maximum(filled - (floor - 0.003), 0.003) ** 2
    else:
        weights = np.ones(len(found), dtype=float)

    order = np.argsort(-weights)
    arr = np.stack([found[i]["probs"] for i in order], axis=0)
    w = weights[order]
    w = w / w.sum()

    topk = min(7, len(found))
    arr_top = arr[:topk]
    w_top = w[:topk]
    w_top = w_top / w_top.sum()

    blend_all = np.tensordot(w, arr, axes=(0, 0))
    blend_top = np.tensordot(w_top, arr_top, axes=(0, 0))
    blend_med = np.median(arr, axis=0)

    legacy = 0.50 * blend_top + 0.35 * blend_all + 0.15 * blend_med
    legacy = np.maximum.accumulate(np.clip(legacy, 0.0, 1.0), axis=1)

    info = {
        "n_files": len(found),
        "top_sub_nums": [int(found[i]["sub_num"]) for i in order[:topk]],
        "top_scores": [None if not np.isfinite(found[i]["score"]) else float(found[i]["score"]) for i in order[:topk]],
    }
    return legacy, info


def legacy_blend_alpha(pred, legacy):
    alpha = np.zeros(pred.shape[1], dtype=float)
    if legacy is None or len(pred) != len(legacy):
        return alpha
    for j in range(pred.shape[1]):
        a = rank01(pred[:, j])
        b = rank01(legacy[:, j])
        corr = np.corrcoef(a, b)[0, 1] if len(a) > 1 else 0.0
        if not np.isfinite(corr):
            corr = 0.0
        mad = float(np.mean(np.abs(pred[:, j] - legacy[:, j])))
        base = 0.04 + 0.10 * np.clip((corr - 0.95) / 0.04, 0.0, 1.0)
        damp = np.clip(1.0 - mad / 0.12, 0.25, 1.0)
        horizon_bonus = [0.00, 0.01, 0.02, 0.03][j]
        alpha[j] = min(0.18, (base + horizon_bonus) * damp)
    return alpha


def build_strata(df: pd.DataFrame) -> np.ndarray:
    t_proxy = np.where(df["event"].values == 1, df["time_to_hit_hours"].values, 72.0)
    q = pd.qcut(t_proxy, q=min(5, max(2, len(df) // 30)), duplicates="drop", labels=False)
    q = np.asarray(q).astype(int)
    return np.array([f"{e}_{qq}" for e, qq in zip(df["event"].values.astype(int), q)], dtype=object)


def harrell_c_index(event, time, risk):
    event = np.asarray(event).astype(bool)
    time = np.asarray(time, dtype=float)
    risk = np.asarray(risk, dtype=float)

    concordant = 0.0
    ties = 0.0
    comparable = 0.0
    n = len(time)

    for i in range(n):
        ti = time[i]
        ei = event[i]
        ri = risk[i]
        for j in range(i + 1, n):
            tj = time[j]
            ej = event[j]
            rj = risk[j]

            if ei and ej:
                if ti == tj:
                    continue
                comparable += 1.0
                if ti < tj:
                    if ri > rj:
                        concordant += 1.0
                    elif ri == rj:
                        ties += 1.0
                else:
                    if rj > ri:
                        concordant += 1.0
                    elif ri == rj:
                        ties += 1.0
            elif ei and (ti < tj):
                comparable += 1.0
                if ri > rj:
                    concordant += 1.0
                elif ri == rj:
                    ties += 1.0
            elif ej and (tj < ti):
                comparable += 1.0
                if rj > ri:
                    concordant += 1.0
                elif ri == rj:
                    ties += 1.0

    if comparable == 0:
        return 0.5
    return (concordant + 0.5 * ties) / comparable


def brier_at_horizon(event, time, pred, horizon):
    event = np.asarray(event).astype(int)
    time = np.asarray(time, dtype=float)
    pred = np.asarray(pred, dtype=float)

    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(float)

    if valid.sum() == 0:
        return np.nan
    return np.mean((pred[valid] - y[valid]) ** 2)


def weighted_brier(event, time, prob_24, prob_48, prob_72):
    b24 = brier_at_horizon(event, time, prob_24, 24)
    b48 = brier_at_horizon(event, time, prob_48, 48)
    b72 = brier_at_horizon(event, time, prob_72, 72)
    return 0.3 * b24 + 0.4 * b48 + 0.3 * b72


def hybrid_score(event, time, prob_24, prob_48, prob_72):
    c = harrell_c_index(event, time, prob_72)
    wb = weighted_brier(event, time, prob_24, prob_48, prob_72)
    return 0.3 * c + 0.7 * (1.0 - wb), c, wb


def make_horizon_target(df: pd.DataFrame, horizon: int):
    event = df["event"].values.astype(int)
    time = df["time_to_hit_hours"].values.astype(float)
    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(int)
    return valid, y


def class_weights_sqrt(y):
    y = np.asarray(y).astype(int)
    pos = y.sum()
    neg = len(y) - pos
    if pos == 0 or neg == 0:
        return np.ones(len(y), dtype=float)
    w_pos = math.sqrt(neg / max(pos, 1))
    return np.where(y == 1, w_pos, 1.0).astype(float)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    num = out.select_dtypes(include=[np.number]).columns
    out[num] = out[num].astype(float)

    out["area_last_ha"] = (out["area_first_ha"].fillna(0).clip(lower=0) + out["area_growth_abs_0_5h"].fillna(0)).clip(lower=0)
    out["area_last_m2"] = out["area_last_ha"] * 10000.0

    out["radius_first_m"] = np.sqrt(np.maximum(out["area_first_ha"].fillna(0).clip(lower=0) * 10000.0, 0.0) / np.pi)
    out["radius_last_m"] = np.sqrt(np.maximum(out["area_last_m2"], 0.0) / np.pi)
    out["radius_gain_m"] = out["radius_last_m"] - out["radius_first_m"]
    out["radius_rate_calc_mph"] = safe_div(out["radius_gain_m"].values, out["dt_first_last_0_5h"].replace(0, np.nan).values, 0.0)
    out["radius_rate_blend_mph"] = 0.5 * out["radius_rate_calc_mph"] + 0.5 * out["radial_growth_rate_m_per_h"].fillna(0)

    out["dist_km"] = out["dist_min_ci_0_5h"] / 1000.0
    out["log_dist"] = np.log1p(np.maximum(out["dist_min_ci_0_5h"], 0.0))
    out["inv_dist_km"] = 1.0 / (1.0 + np.maximum(out["dist_km"], 0.0))
    out["near_10km"] = (out["dist_min_ci_0_5h"] <= 10000).astype(int)
    out["near_20km"] = (out["dist_min_ci_0_5h"] <= 20000).astype(int)
    out["near_50km"] = (out["dist_min_ci_0_5h"] <= 50000).astype(int)

    out["closing_pos"] = out["closing_speed_m_per_h"].fillna(0).clip(lower=0)
    out["along_pos"] = out["along_track_speed"].fillna(0).clip(lower=0)
    out["align_pos"] = out["alignment_cos"].fillna(0).clip(lower=0)
    out["growth_pos"] = out["area_growth_rate_ha_per_h"].fillna(0).clip(lower=0)
    out["radial_pos"] = out["radius_rate_blend_mph"].clip(lower=0)
    out["centroid_pos"] = out["centroid_speed_m_per_h"].fillna(0).clip(lower=0)

    out["effective_forward_speed"] = (
        0.50 * out["closing_pos"] +
        0.20 * out["along_pos"] +
        0.60 * out["radial_pos"] +
        0.15 * out["centroid_pos"] * out["alignment_abs"].fillna(0) +
        0.10 * safe_div(np.maximum(out["projected_advance_m"], 0.0).values, np.maximum(out["dt_first_last_0_5h"], 0.25).values, 0.0)
    )
    out["effective_forward_speed"] = np.maximum(out["effective_forward_speed"], 0.0)

    out["margin_to_5km_m"] = out["dist_min_ci_0_5h"] - 5000.0 - out["radius_last_m"]
    out["margin_to_centroid_m"] = out["dist_min_ci_0_5h"] - out["radius_last_m"]
    out["wavefront_eta_h"] = np.where(
        out["effective_forward_speed"] > 50.0,
        out["margin_to_5km_m"] / np.maximum(out["effective_forward_speed"], 50.0),
        999.0
    )
    out["wavefront_eta_h"] = np.clip(out["wavefront_eta_h"], -24.0, 240.0)
    out["wavefront_eta_log"] = np.sign(out["wavefront_eta_h"]) * np.log1p(np.abs(out["wavefront_eta_h"]))

    out["threat_gravity"] = (
        (out["radius_last_m"] + np.maximum(out["projected_advance_m"], 0.0) + 15.0 * np.maximum(out["growth_pos"], 0.0)) *
        (0.5 + out["alignment_abs"].fillna(0)) /
        (500.0 + np.maximum(out["dist_min_ci_0_5h"], 0.0))
    )
    out["aligned_advance"] = out["alignment_abs"].fillna(0) * out["closing_pos"]
    out["aligned_growth"] = out["alignment_abs"].fillna(0) * out["radial_pos"]
    out["threat_score"] = (
        2.0 * out["inv_dist_km"] +
        0.003 * out["closing_pos"] +
        0.003 * out["radial_pos"] +
        0.8 * out["alignment_abs"].fillna(0) +
        0.05 * out["log1p_area_first"].fillna(0) +
        0.03 * out["log1p_growth"].fillna(0)
    )
    out["near_score"] = expit(-(out["margin_to_5km_m"] / 7000.0))
    out["fast_score"] = expit((out["effective_forward_speed"] - 300.0) / 150.0)
    out["urgent_score"] = 0.60 * out["near_score"] + 0.40 * out["fast_score"]
    out["quality_score"] = 1.0 - 0.35 * out["low_temporal_resolution_0_5h"].fillna(0)
    out["risk_x_quality"] = out["urgent_score"] * out["quality_score"]

    for h, s in zip(HORIZONS, [3.5, 5.0, 8.0, 10.0]):
        out[f"eta_sigmoid_{h}h"] = expit((h - out["wavefront_eta_h"]) / s)

    for col, period in [("event_start_hour", 24), ("event_start_dayofweek", 7), ("event_start_month", 12)]:
        if col in out.columns:
            out[f"{col}_sin"] = np.sin(2 * np.pi * out[col] / period)
            out[f"{col}_cos"] = np.cos(2 * np.pi * out[col] / period)

    out["dist_x_align"] = out["dist_km"] * out["alignment_abs"].fillna(0)
    out["close_x_align"] = out["closing_pos"] * out["alignment_abs"].fillna(0)
    out["margin_x_growth"] = out["margin_to_5km_m"] * out["radial_pos"]
    out["lowres_x_margin"] = out["low_temporal_resolution_0_5h"].fillna(0) * out["margin_to_5km_m"]
    out["firstlast_density"] = safe_div(out["num_perimeters_0_5h"].values, np.maximum(out["dt_first_last_0_5h"], 0.25).values, 0.0)

    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    return out


def expand_discrete_time(X_df: pd.DataFrame, time_arr: np.ndarray, event_arr: np.ndarray) -> pd.DataFrame:
    rows = []
    targets = []
    orig_rows = []

    for idx in range(len(X_df)):
        t = float(time_arr[idx])
        e = int(event_arr[idx])

        for j, (start, end) in enumerate(INTERVALS):
            if e == 1:
                if t <= start + EPS:
                    break
                row = X_df.iloc[idx].copy()
                row["interval_idx"] = str(j)
                row["interval_start"] = float(start)
                row["interval_end"] = float(end)
                row["interval_len"] = float(end - start)
                rows.append(row)
                targets.append(1 if t <= end + EPS else 0)
                orig_rows.append(idx)
                if t <= end + EPS:
                    break
            else:
                if end <= t + EPS:
                    row = X_df.iloc[idx].copy()
                    row["interval_idx"] = str(j)
                    row["interval_start"] = float(start)
                    row["interval_end"] = float(end)
                    row["interval_len"] = float(end - start)
                    rows.append(row)
                    targets.append(0)
                    orig_rows.append(idx)
                else:
                    break

    exp_df = pd.DataFrame(rows)
    exp_df["hazard_target"] = np.array(targets, dtype=int)
    exp_df["orig_row"] = np.array(orig_rows, dtype=int)
    return exp_df


def make_interval_prediction_frame(X_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for idx in range(len(X_df)):
        base = X_df.iloc[idx]
        for j, (start, end) in enumerate(INTERVALS):
            row = base.copy()
            row["interval_idx"] = str(j)
            row["interval_start"] = float(start)
            row["interval_end"] = float(end)
            row["interval_len"] = float(end - start)
            rows.append(row)
    out = pd.DataFrame(rows)
    return out


def constant_predict(y_train, n_val, n_test):
    p = float(np.mean(y_train)) if len(y_train) else 0.5
    return np.full(n_val, p, dtype=float), np.full(n_test, p, dtype=float)


def fit_predict_lgb(X_train, y_train, X_val, X_test, seed):
    if (not HAVE_LGB) or len(np.unique(y_train)) < 2:
        return constant_predict(y_train, len(X_val), len(X_test))
    sw = class_weights_sqrt(y_train)
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=360,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=10,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=4.0,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(X_train, y_train, sample_weight=sw)
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]


def fit_predict_cat(X_train, y_train, X_val, X_test, seed):
    if (not HAVE_CAT) or len(np.unique(y_train)) < 2:
        return constant_predict(y_train, len(X_val), len(X_test))
    sw = class_weights_sqrt(y_train)
    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=450,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=8.0,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )
    model.fit(X_train, y_train, sample_weight=sw)
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]


def fit_predict_xgb(X_train, y_train, X_val, X_test, seed):
    if (not HAVE_XGB) or len(np.unique(y_train)) < 2:
        return constant_predict(y_train, len(X_val), len(X_test))
    sw = class_weights_sqrt(y_train)
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        n_estimators=360,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=2.0,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=4.0,
        gamma=0.0,
        random_state=seed,
        n_jobs=-1,
        tree_method="hist",
        eval_metric="logloss",
        enable_categorical=False,
    )
    model.fit(X_train, y_train, sample_weight=sw)
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]


def fit_predict_etr(X_train, y_train, X_val, X_test, seed):
    if len(np.unique(y_train)) < 2:
        return constant_predict(y_train, len(X_val), len(X_test))
    sw = class_weights_sqrt(y_train)
    model = ExtraTreesClassifier(
        n_estimators=600,
        max_depth=8,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(X_train, y_train, sample_weight=sw)
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]


def fit_predict_logit(X_train, y_train, X_val, X_test, seed):
    if len(np.unique(y_train)) < 2:
        return constant_predict(y_train, len(X_val), len(X_test))
    sw = class_weights_sqrt(y_train)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xva = scaler.transform(X_val)
    Xte = scaler.transform(X_test)
    model = LogisticRegression(
        C=0.7,
        max_iter=5000,
        solver="lbfgs",
        random_state=seed,
    )
    model.fit(Xtr, y_train, sample_weight=sw)
    return model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]


def fit_predict_hgb(X_train, y_train, X_val, X_test, seed):
    if len(np.unique(y_train)) < 2:
        return constant_predict(y_train, len(X_val), len(X_test))
    sw = class_weights_sqrt(y_train)
    model = HistGradientBoostingClassifier(
        learning_rate=0.04,
        max_iter=320,
        max_depth=3,
        max_leaf_nodes=15,
        min_samples_leaf=10,
        l2_regularization=1.0,
        random_state=seed,
    )
    model.fit(X_train, y_train, sample_weight=sw)
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]


def fit_predict_hazard_cat(X_train_df, time_train, event_train, X_val_df, X_test_df, seed):
    if not HAVE_CAT:
        base = np.mean(event_train)
        out_val = np.tile(base, (len(X_val_df), len(HORIZONS)))
        out_test = np.tile(base, (len(X_test_df), len(HORIZONS)))
        return out_val, out_test

    exp_train = expand_discrete_time(X_train_df, time_train, event_train)
    if exp_train["hazard_target"].nunique() < 2:
        base = float(exp_train["hazard_target"].mean()) if len(exp_train) else float(np.mean(event_train))
        out_val = np.tile(base, (len(X_val_df), len(HORIZONS)))
        out_test = np.tile(base, (len(X_test_df), len(HORIZONS)))
        return out_val, out_test

    cat_cols = ["interval_idx"]
    X_cols = [c for c in exp_train.columns if c not in ["hazard_target", "orig_row"]]
    sw = class_weights_sqrt(exp_train["hazard_target"].values)

    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=380,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=8.0,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )
    model.fit(exp_train[X_cols], exp_train["hazard_target"], cat_features=cat_cols, sample_weight=sw)

    val_exp = make_interval_prediction_frame(X_val_df)
    test_exp = make_interval_prediction_frame(X_test_df)

    h_val = model.predict_proba(val_exp[X_cols])[:, 1].reshape(len(X_val_df), len(HORIZONS))
    h_test = model.predict_proba(test_exp[X_cols])[:, 1].reshape(len(X_test_df), len(HORIZONS))

    p_val = 1.0 - np.cumprod(1.0 - clip_prob(h_val), axis=1)
    p_test = 1.0 - np.cumprod(1.0 - clip_prob(h_test), axis=1)
    return p_val, p_test


def fit_predict_hazard_logit(X_train_df, time_train, event_train, X_val_df, X_test_df, seed):
    exp_train = expand_discrete_time(X_train_df, time_train, event_train)

    if len(exp_train) == 0 or exp_train["hazard_target"].nunique() < 2:
        base = float(np.mean(event_train))
        out_val = np.tile(base, (len(X_val_df), len(HORIZONS)))
        out_test = np.tile(base, (len(X_test_df), len(HORIZONS)))
        return out_val, out_test

    X_cols = [c for c in exp_train.columns if c not in ["hazard_target", "orig_row"]]
    num_train = exp_train[X_cols].copy()

    interval_dummies = pd.get_dummies(num_train["interval_idx"], prefix="int")
    num_train = num_train.drop(columns=["interval_idx"])
    num_train = pd.concat([num_train.reset_index(drop=True), interval_dummies.reset_index(drop=True)], axis=1)

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(num_train)
    ytr = exp_train["hazard_target"].values
    sw = class_weights_sqrt(ytr)

    model = LogisticRegression(
        C=0.7,
        max_iter=5000,
        solver="lbfgs",
        random_state=seed,
    )
    model.fit(Xtr, ytr, sample_weight=sw)

    def pred_cum(X_df):
        exp_df = make_interval_prediction_frame(X_df)
        interval_dummies_ = pd.get_dummies(exp_df["interval_idx"], prefix="int")
        num_df = exp_df.drop(columns=["interval_idx"])
        interval_dummies_ = interval_dummies_.reindex(columns=interval_dummies.columns, fill_value=0)
        num_df = pd.concat([num_df.reset_index(drop=True), interval_dummies_.reset_index(drop=True)], axis=1)
        Xs = scaler.transform(num_df)
        hz = model.predict_proba(Xs)[:, 1].reshape(len(X_df), len(HORIZONS))
        return 1.0 - np.cumprod(1.0 - clip_prob(hz), axis=1)

    return pred_cum(X_val_df), pred_cum(X_test_df)


def fit_predict_rank_cat(X_train, time_train, event_train, X_val, X_test, seed):
    if not HAVE_CAT:
        base_score = -np.log1p(time_train) + 0.35 * event_train
        return np.full(len(X_val), float(np.mean(base_score))), np.full(len(X_test), float(np.mean(base_score)))

    target = -np.log1p(time_train) + 0.35 * event_train
    sw = np.where(event_train == 1, 1.0, 0.75).astype(float)

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=420,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=8.0,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )
    model.fit(X_train, target, sample_weight=sw)
    return model.predict(X_val), model.predict(X_test)


def fit_isotonic_or_constant(x_train, y_train, x_apply_train, x_apply_test):
    x_train = np.asarray(x_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    x_apply_train = np.asarray(x_apply_train, dtype=float)
    x_apply_test = np.asarray(x_apply_test, dtype=float)

    if len(np.unique(y_train)) < 2:
        p = float(np.mean(y_train)) if len(y_train) else 0.5
        return np.full(len(x_apply_train), p), np.full(len(x_apply_test), p)

    if len(np.unique(x_train)) < 8:
        p = float(np.mean(y_train))
        return np.full(len(x_apply_train), p), np.full(len(x_apply_test), p)

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(x_train, y_train)
    return clip_prob(iso.predict(x_apply_train)), clip_prob(iso.predict(x_apply_test))


def optimize_simplex_blend(P, objective, force_include_last=True):
    P = np.asarray(P, dtype=float)
    n_models = P.shape[1]

    if n_models == 1:
        return np.array([1.0], dtype=float)

    w0 = np.ones(n_models, dtype=float) / n_models
    bounds = [(0.0, 1.0)] * n_models
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    res = minimize(
        fun=lambda w: objective(np.clip(P @ w, 0.0, 1.0)),
        x0=w0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"maxiter": 1000, "ftol": 1e-10, "disp": False},
    )

    if not res.success:
        return w0
    w = np.clip(res.x, 0.0, 1.0)
    if w.sum() <= 0:
        return w0
    return w / w.sum()


def select_candidates(pred_dict, event, time, horizon, max_nonconst=6):
    names = [n for n in pred_dict.keys() if n != "const"]
    scores = []

    for name in names:
        p = pred_dict[name]
        if horizon == 72:
            c = harrell_c_index(event, time, p)
            b = brier_at_horizon(event, time, p, 72)
            score = 0.59 * c + 0.41 * (1.0 - b)
            scores.append((name, score))
        else:
            b = brier_at_horizon(event, time, p, horizon)
            scores.append((name, -b))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    selected = [n for n, _ in scores[:max_nonconst]]
    selected.append("const")
    return selected


def fit_blend_for_horizon(train_pred_dict, test_pred_dict, event, time, horizon, strat_feature=None, optimize_hybrid72_context=None):
    selected = select_candidates(train_pred_dict, event, time, horizon, max_nonconst=7)

    P_train = np.column_stack([train_pred_dict[n] for n in selected])
    P_test = np.column_stack([test_pred_dict[n] for n in selected])

    def objective_for_horizon(pred):
        pred = clip_prob(pred)
        if horizon == 72 and optimize_hybrid72_context is not None:
            p24_fix, p48_fix = optimize_hybrid72_context
            score, _, _ = hybrid_score(event, time, p24_fix, p48_fix, pred)
            return -score
        return brier_at_horizon(event, time, pred, horizon)

    w_global = optimize_simplex_blend(P_train, objective_for_horizon)
    pred_train_best = np.clip(P_train @ w_global, 0.0, 1.0)
    pred_test_best = np.clip(P_test @ w_global, 0.0, 1.0)
    best_obj = objective_for_horizon(pred_train_best)
    best_info = {
        "mode": "global",
        "selected": selected,
        "weights": dict(zip(selected, w_global.round(6))),
    }

    if strat_feature is not None:
        thr_vals = np.unique(np.quantile(strat_feature, [0.20, 0.35, 0.50, 0.65, 0.80]))
        for thr in thr_vals:
            near = strat_feature <= thr
            far = ~near

            valid_near = near.copy()
            valid_far = far.copy()

            if horizon != 72:
                event_arr = np.asarray(event).astype(int)
                time_arr = np.asarray(time).astype(float)
                valid_mask = ~((event_arr == 0) & (time_arr < horizon))
                valid_near = near & valid_mask
                valid_far = far & valid_mask

            if valid_near.sum() < 20 or valid_far.sum() < 20:
                continue

            if horizon == 72 and optimize_hybrid72_context is not None:
                def obj_near(pred):
                    return brier_at_horizon(event[valid_near], time[valid_near], pred, 72)

                def obj_far(pred):
                    return brier_at_horizon(event[valid_far], time[valid_far], pred, 72)
            else:
                def obj_near(pred):
                    return brier_at_horizon(event[valid_near], time[valid_near], pred, horizon)

                def obj_far(pred):
                    return brier_at_horizon(event[valid_far], time[valid_far], pred, horizon)

            w_near = optimize_simplex_blend(P_train[valid_near], obj_near)
            w_far = optimize_simplex_blend(P_train[valid_far], obj_far)

            pred_train = np.where(
                near,
                np.clip(P_train @ w_near, 0.0, 1.0),
                np.clip(P_train @ w_far, 0.0, 1.0),
            )
            pred_test = np.where(
                strat_feature_test <= thr,
                np.clip(P_test @ w_near, 0.0, 1.0),
                np.clip(P_test @ w_far, 0.0, 1.0),
            )

            obj = objective_for_horizon(pred_train)
            if obj < best_obj:
                best_obj = obj
                pred_train_best = pred_train
                pred_test_best = pred_test
                best_info = {
                    "mode": "stratified",
                    "threshold": float(thr),
                    "selected": selected,
                    "weights_near": dict(zip(selected, w_near.round(6))),
                    "weights_far": dict(zip(selected, w_far.round(6))),
                }

    return clip_prob(pred_train_best), clip_prob(pred_test_best), best_info


set_seed(42)
DATA_DIR = find_data_dir()

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

train_fe = add_features(train)
test_fe = add_features(test)

target_cols = ["event_id", "time_to_hit_hours", "event"]
feature_cols = [c for c in train_fe.columns if c not in target_cols]

imputer = SimpleImputer(strategy="median")
train_X = pd.DataFrame(imputer.fit_transform(train_fe[feature_cols]), columns=feature_cols, index=train_fe.index).astype(np.float32)
test_X = pd.DataFrame(imputer.transform(test_fe[feature_cols]), columns=feature_cols, index=test_fe.index).astype(np.float32)

event = train["event"].astype(int).values
time_hit = train["time_to_hit_hours"].astype(float).values

strata = build_strata(train)
n_train = len(train)
n_test = len(test)

model_names = [
    "lgb",
    "cat",
    "xgb",
    "etr",
    "logit",
    "hgb",
    "haz_cat",
    "haz_logit",
]
oof = {name: np.zeros((n_train, len(HORIZONS)), dtype=float) for name in model_names}
test_pred = {name: np.zeros((n_test, len(HORIZONS)), dtype=float) for name in model_names}

rank_oof = np.zeros(n_train, dtype=float)
rank_test = np.zeros(n_test, dtype=float)
physics_score_train = (
    0.65 * (-train_X["wavefront_eta_h"].values) +
    0.20 * rank01(train_X["threat_gravity"].values) +
    0.15 * train_X["urgent_score"].values
)
physics_score_test = (
    0.65 * (-test_X["wavefront_eta_h"].values) +
    0.20 * rank01(test_X["threat_gravity"].values) +
    0.15 * test_X["urgent_score"].values
)

row_counts = np.zeros(n_train, dtype=float)

fold_records = []

for seed in SEEDS:
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_X, strata), start=1):
        X_tr = train_X.iloc[tr_idx].copy()
        X_va = train_X.iloc[va_idx].copy()
        X_te = test_X.copy()

        y_event_tr = event[tr_idx]
        y_event_va = event[va_idx]
        y_time_tr = time_hit[tr_idx]
        y_time_va = time_hit[va_idx]

        # horizon-specific models
        for h_idx, H in enumerate(HORIZONS):
            tr_valid, y_tr_h = make_horizon_target(train.iloc[tr_idx], H)
            va_valid, y_va_h = make_horizon_target(train.iloc[va_idx], H)

            Xtr_h = X_tr.iloc[tr_valid]
            Xva_eval = X_va.iloc[va_valid]

            ytr_h = y_tr_h[tr_valid]
            yva_h = y_va_h[va_valid]

            # lgb
            p_va, p_te = fit_predict_lgb(Xtr_h, ytr_h, X_va, X_te, seed + h_idx)
            oof["lgb"][va_idx, h_idx] += p_va
            test_pred["lgb"][:, h_idx] += p_te / (len(SEEDS) * N_SPLITS)

            # cat
            p_va, p_te = fit_predict_cat(Xtr_h, ytr_h, X_va, X_te, seed + 100 + h_idx)
            oof["cat"][va_idx, h_idx] += p_va
            test_pred["cat"][:, h_idx] += p_te / (len(SEEDS) * N_SPLITS)

            # xgb
            p_va, p_te = fit_predict_xgb(Xtr_h, ytr_h, X_va, X_te, seed + 150 + h_idx)
            oof["xgb"][va_idx, h_idx] += p_va
            test_pred["xgb"][:, h_idx] += p_te / (len(SEEDS) * N_SPLITS)

            # etr
            p_va, p_te = fit_predict_etr(Xtr_h, ytr_h, X_va, X_te, seed + 200 + h_idx)
            oof["etr"][va_idx, h_idx] += p_va
            test_pred["etr"][:, h_idx] += p_te / (len(SEEDS) * N_SPLITS)

            # logit
            p_va, p_te = fit_predict_logit(Xtr_h, ytr_h, X_va, X_te, seed + 300 + h_idx)
            oof["logit"][va_idx, h_idx] += p_va
            test_pred["logit"][:, h_idx] += p_te / (len(SEEDS) * N_SPLITS)

            # hgb
            p_va, p_te = fit_predict_hgb(Xtr_h, ytr_h, X_va, X_te, seed + 400 + h_idx)
            oof["hgb"][va_idx, h_idx] += p_va
            test_pred["hgb"][:, h_idx] += p_te / (len(SEEDS) * N_SPLITS)

        # hazard models
        p_va_mat, p_te_mat = fit_predict_hazard_cat(X_tr, y_time_tr, y_event_tr, X_va, X_te, seed + 500)
        oof["haz_cat"][va_idx] += p_va_mat
        test_pred["haz_cat"] += p_te_mat / (len(SEEDS) * N_SPLITS)

        p_va_mat, p_te_mat = fit_predict_hazard_logit(X_tr, y_time_tr, y_event_tr, X_va, X_te, seed + 600)
        oof["haz_logit"][va_idx] += p_va_mat
        test_pred["haz_logit"] += p_te_mat / (len(SEEDS) * N_SPLITS)

        # ranking model
        s_va, s_te = fit_predict_rank_cat(X_tr, y_time_tr, y_event_tr, X_va, X_te, seed + 700)
        rank_oof[va_idx] += s_va
        rank_test += s_te / (len(SEEDS) * N_SPLITS)

        row_counts[va_idx] += 1.0

        gc.collect()

for name in model_names:
    oof[name] = oof[name] / row_counts.reshape(-1, 1)

rank_oof = rank_oof / row_counts

# calibrated auxiliary candidates
rank_prob_train = np.zeros((n_train, len(HORIZONS)), dtype=float)
rank_prob_test = np.zeros((n_test, len(HORIZONS)), dtype=float)
physics_prob_train = np.zeros((n_train, len(HORIZONS)), dtype=float)
physics_prob_test = np.zeros((n_test, len(HORIZONS)), dtype=float)

for h_idx, H in enumerate(HORIZONS):
    valid_h, y_h = make_horizon_target(train, H)

    p_tr, p_te = fit_isotonic_or_constant(
        x_train=rank_oof[valid_h],
        y_train=y_h[valid_h],
        x_apply_train=rank_oof,
        x_apply_test=rank_test,
    )
    rank_prob_train[:, h_idx] = p_tr
    rank_prob_test[:, h_idx] = p_te

    p_tr, p_te = fit_isotonic_or_constant(
        x_train=physics_score_train[valid_h],
        y_train=y_h[valid_h],
        x_apply_train=physics_score_train,
        x_apply_test=physics_score_test,
    )
    physics_prob_train[:, h_idx] = p_tr
    physics_prob_test[:, h_idx] = p_te

# constant candidates
const_train = np.zeros((n_train, len(HORIZONS)), dtype=float)
const_test = np.zeros((n_test, len(HORIZONS)), dtype=float)
const1_train = np.ones((n_train, len(HORIZONS)), dtype=float)
const1_test = np.ones((n_test, len(HORIZONS)), dtype=float)
for h_idx, H in enumerate(HORIZONS):
    valid_h, y_h = make_horizon_target(train, H)
    base_rate = float(y_h[valid_h].mean()) if valid_h.sum() else float(y_h.mean())
    const_train[:, h_idx] = base_rate
    const_test[:, h_idx] = base_rate

# candidate dicts by horizon
train_candidates = {}
test_candidates = {}
for h_idx, H in enumerate(HORIZONS):
    train_candidates[H] = {
        "lgb": oof["lgb"][:, h_idx],
        "cat": oof["cat"][:, h_idx],
        "xgb": oof["xgb"][:, h_idx],
        "etr": oof["etr"][:, h_idx],
        "logit": oof["logit"][:, h_idx],
        "hgb": oof["hgb"][:, h_idx],
        "haz_cat": oof["haz_cat"][:, h_idx],
        "haz_logit": oof["haz_logit"][:, h_idx],
        "rank": rank_prob_train[:, h_idx],
        "physics": physics_prob_train[:, h_idx],
        "const": const_train[:, h_idx],
        "const1": const1_train[:, h_idx],
    }
    test_candidates[H] = {
        "lgb": test_pred["lgb"][:, h_idx],
        "cat": test_pred["cat"][:, h_idx],
        "xgb": test_pred["xgb"][:, h_idx],
        "etr": test_pred["etr"][:, h_idx],
        "logit": test_pred["logit"][:, h_idx],
        "hgb": test_pred["hgb"][:, h_idx],
        "haz_cat": test_pred["haz_cat"][:, h_idx],
        "haz_logit": test_pred["haz_logit"][:, h_idx],
        "rank": rank_prob_test[:, h_idx],
        "physics": physics_prob_test[:, h_idx],
        "const": const_test[:, h_idx],
        "const1": const1_test[:, h_idx],
    }

margin_train = train_X["margin_to_5km_m"].values
margin_test = test_X["margin_to_5km_m"].values
strat_feature_test = margin_test  # used inside blend routine

# optimize blends in sequence
blend_info = {}
final_oof = np.zeros((n_train, len(HORIZONS)), dtype=float)
final_test = np.zeros((n_test, len(HORIZONS)), dtype=float)

# 12h
final_oof[:, 0], final_test[:, 0], blend_info["12h"] = fit_blend_for_horizon(
    train_candidates[12], test_candidates[12], event, time_hit, 12, strat_feature=margin_train
)

# 24h
final_oof[:, 1], final_test[:, 1], blend_info["24h"] = fit_blend_for_horizon(
    train_candidates[24], test_candidates[24], event, time_hit, 24, strat_feature=margin_train
)

# 48h
final_oof[:, 2], final_test[:, 2], blend_info["48h"] = fit_blend_for_horizon(
    train_candidates[48], test_candidates[48], event, time_hit, 48, strat_feature=margin_train
)

# 72h uses hybrid-aware optimization with 24h/48h already fixed
final_oof[:, 3], final_test[:, 3], blend_info["72h"] = fit_blend_for_horizon(
    train_candidates[72],
    test_candidates[72],
    event,
    time_hit,
    72,
    strat_feature=margin_train,
    optimize_hybrid72_context=(final_oof[:, 1], final_oof[:, 2]),
)

# enforce monotonicity
final_oof = np.maximum.accumulate(clip_prob(final_oof), axis=1)
final_test = np.maximum.accumulate(clip_prob(final_test), axis=1)

legacy_test, legacy_info = discover_legacy_submissions(test["event_id"].astype(str).tolist())
if legacy_test is not None:
    alpha = legacy_blend_alpha(final_test, legacy_test)
    print("Legacy submission blend found:", legacy_info)
    print("Legacy blend alpha:", dict(zip(HORIZONS, alpha.round(4))))
    final_test = clip_prob((1.0 - alpha.reshape(1, -1)) * final_test + alpha.reshape(1, -1) * legacy_test)
    final_test = np.maximum.accumulate(final_test, axis=1)

# local scores
score, cidx, wb = hybrid_score(event, time_hit, final_oof[:, 1], final_oof[:, 2], final_oof[:, 3])
print("=" * 80)
print("WiDS 2026 | Physics-Gated Multi-Survival Stack")
print("=" * 80)
print(f"OOF Hybrid Score : {score:.6f}")
print(f"OOF C-index      : {cidx:.6f}")
print(f"OOF WeightedBrier: {wb:.6f}")
for H, idx in zip(HORIZONS, range(4)):
    b = brier_at_horizon(event, time_hit, final_oof[:, idx], H)
    print(f"Brier@{H:>2}h      : {b:.6f}")
print("=" * 80)
print(json.dumps(blend_info, indent=2))

# create outputs
pred_df = pd.DataFrame({
    "event_id": test["event_id"].values,
    "prob_12h": final_test[:, 0],
    "prob_24h": final_test[:, 1],
    "prob_48h": final_test[:, 2],
    "prob_72h": final_test[:, 3],
})

submission = sample_sub[["event_id"]].merge(pred_df, on="event_id", how="left")
assert submission["event_id"].nunique() == len(submission)
assert len(submission) == len(test)
assert submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].notnull().all().all()

probs = submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values
assert np.all((probs >= 0.0) & (probs <= 1.0))
assert np.all(probs[:, 0] <= probs[:, 1] + 1e-12)
assert np.all(probs[:, 1] <= probs[:, 2] + 1e-12)
assert np.all(probs[:, 2] <= probs[:, 3] + 1e-12)

submission.to_csv("submission.csv", index=False)

oof_out = pd.DataFrame({
    "event_id": train["event_id"].values,
    "prob_12h": final_oof[:, 0],
    "prob_24h": final_oof[:, 1],
    "prob_48h": final_oof[:, 2],
    "prob_72h": final_oof[:, 3],
    "time_to_hit_hours": train["time_to_hit_hours"].values,
    "event": train["event"].values,
})
oof_out.to_csv("oof_predictions.csv", index=False)

print("Saved: submission.csv")
print("Saved: oof_predictions.csv")
print(submission.head())


WiDS 2026 | Physics-Gated Multi-Survival Stack
OOF Hybrid Score : 0.970158
OOF C-index      : 0.934995
OOF WeightedBrier: 0.014773
Brier@12h      : 0.052602
Brier@24h      : 0.027820
Brier@48h      : 0.016065
Brier@72h      : 0.000002
{
  "12h": {
    "mode": "stratified",
    "threshold": -2966.392333984375,
    "selected": [
      "rank",
      "xgb",
      "etr",
      "haz_cat",
      "lgb",
      "cat",
      "physics",
      "const"
    ],
    "weights_near": {
      "rank": 0.411713,
      "xgb": 0.0,
      "etr": 0.0,
      "haz_cat": 0.0,
      "lgb": 0.0,
      "cat": 0.0,
      "physics": 0.588287,
      "const": 0.0
    },
    "weights_far": {
      "rank": 0.658334,
      "xgb": 0.341666,
      "etr": 0.0,
      "haz_cat": 0.0,
      "lgb": 0.0,
      "cat": 0.0,
      "physics": 0.0,
      "const": 0.0
    }
  },
  "24h": {
    "mode": "stratified",
    "threshold": -2966.392333984375,
    "selected": [
      "hgb",
      "xgb",
      "rank",
      "haz_cat",
      "lgb",